In [ ]:
import pandas as pd
import time

def process_survey_data():
    # Track execution time
    start_time = time.time()
    
    # 1. Load only necessary columns to save memory
    print("Loading CSV with selected columns...")
    use_cols = ['ResponseId', 'Age', 'RemoteWork', 'EdLevel']
    df = pd.read_csv(
        "../Abgaben/survey_results_public.csv",
        usecols=use_cols,
        index_col="ResponseId"
    )
    print(f"CSV loaded. Shape: {df.shape}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    
    # 2. Process EdLevel - more efficient string handling
    print("\nProcessing education levels...")
    df["EdLevel"] = df["EdLevel"].str.extract(r'^([^(]+)').squeeze().str.strip()
    
    # 3. Map RemoteWork to numerical values
    print("Mapping remote work categories...")
    remote_map = {
        "Remote": 0,
        "In-person": 1,
        "Hybrid (some remote, leans heavy to in-person)": 0.75,
        "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
        "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
    }
    df["RemoteCategoryNum"] = df["RemoteWork"].map(remote_map)
    
    # 4. Map Age to numerical values
    print("Processing age categories...")
    age_map = {
        "Under 18 years old": 17,
        "18-24 years old": 21,
        "25-34 years old": 29,
        "35-44 years old": 39,
        "45-54 years old": 49,
        "55-64 years old": 59,
        "65 years or older": 70
    }
    df["AgeNum"] = df['Age'].map(age_map)
    
    # 5. Filter out ages over 65
    print("Filtering data...")
    df = df[df['AgeNum'] <= 65].copy()
    
    # 6. Save to CSV
    print("Saving to CSV...")
    df.to_csv("survey_results_shortened.csv")
    
    # Print summary
    print("\nProcessing complete!")
    print(f"Final DataFrame shape: {df.shape}")
    print(f"Total execution time: {time.time() - start_time:.2f} seconds")
    
    return df

# Run the processing
df_processed = process_survey_data()

In [ ]:
#Werte Kategorisieren um später besser auf numerische Werte mappen zu können

remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
# df.drop(columns=["RemoteCategory"], inplace=True)
df.insert(
    df.columns.get_loc("RemoteWork") + 1,
    "RemoteCategoryNum",
    df["RemoteWork"].map(remote_map)
)
df

In [ ]:
df['RemoteCategoryNum'].describe()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}

df.insert(
    df.columns.get_loc("Age") + 1,
    "AgeNum",
    df['Age'].map(age_map)
)


In [ ]:
cols = ["AgeNum", "YearsCode", "WorkExp", "JobSat", "RemoteCategoryNum"]
df_corr = df[cols]
df_corr

In [ ]:
sns.heatmap(df_corr.corr(method='spearman'),annot=True, cmap="coolwarm", vmin=-1, vmax=1)

- Korrelation zw. AgeNum, YearsCode und WorkExp
- fast keine Korrelation zw. JobSat und Alter bzw. YearsCode & WorkExp
    - Jobzufriedenheit hängt dementsprechend nicht vom Alter, YearsCode oder WorkExp ab
- ältere & erfahrenere Entwickler bevorzugen eher Remote
- jüngere Entwickler bevorzugen eher In-Person oder In-Person-heavy Hybrid
    - weil RemoteCategoryNum bei ihnen tendenziell höher ist
- Remote-Präferenz hängt ganz leicht mit JobSat zusammen

In [ ]:
currency_counts = df["Currency"].value_counts(dropna=False)

currency_counts